# What This Walkthrough Does

This walkthrough introduces static web scraping with `requests` and `BeautifulSoup`.

The main first example is the same substantive source used on Day 1: a Wikipedia page about the Digital Services Act. This keeps the object constant while the access route changes:

- Day 1: MediaWiki API returns structured JSON.
- Day 2: the Wikipedia website returns HTML.

After that, we keep `quotes.toscrape.com` as a deliberately clean practice page for repeated records, and then move to the MethodsNET exercise separately.

By the end, you should understand how to:

- check `robots.txt` as a first access signal
- fetch raw HTML
- save raw HTML before parsing
- inspect page structure
- use CSS selectors
- extract document text, headings, and links
- extract repeated records from a clean practice page
- resolve relative URLs
- save processed tables

# Imports and Shared Setup


## Packages Used in This Notebook

This walkthrough uses:

- `requests` to fetch the raw HTML page
- `BeautifulSoup` to parse and search the HTML
- `pandas` to store extracted records as tables
- `RobotFileParser` to inspect `robots.txt` as a first access signal
- `urlparse` and `urljoin` to work with URLs
- `Path` to create output folders and file paths

The key division of labor is: `requests` fetches, `BeautifulSoup` parses, `pandas` tabulates.


In [ ]:
# Path helps create output folders and file paths.
from pathlib import Path
# pprint makes long strings or nested objects easier to inspect during teaching.
from pprint import pprint
# urljoin resolves relative links; urlparse splits a URL into scheme/domain/path parts.
from urllib.parse import urljoin, urlparse
# RobotFileParser reads robots.txt rules as a first access signal.
from urllib.robotparser import RobotFileParser

# pandas turns extracted rows into tables.
import pandas as pd
# requests fetches the raw HTML from the server.
import requests
# BeautifulSoup parses HTML and lets us search it with selectors.
from bs4 import BeautifulSoup


In [ ]:
# All examples in this notebook write into the same teaching data folder.
outdir = Path("data")
raw_dir = outdir / "raw"
processed_dir = outdir / "processed"

# mkdir(..., parents=True, exist_ok=True) creates folders and does not fail if they already exist.
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)


# First Static Workflow: Wikipedia Article

We start with Wikipedia because Day 1 used the MediaWiki API. This makes the comparison clear: same object, different access route.

With the API, we asked for structured JSON. With static scraping, we ask the website for HTML and then decide which elements to extract.

This example is useful because a Wikipedia article is not a list of neat cards. It has article text, headings, links, references, navigation, sidebars, and page metadata. That forces a methodological question:

> Which parts of the page count as the data we want?

For Wikipedia, the API is often better for reproducible article extraction. Here we use HTML scraping because Day 2 is about understanding web page structure.


## Why Not Start with a News Outlet?

News pages are realistic, but they are often difficult first classroom examples:

- paywalls and consent banners can block access
- HTML structures change frequently
- article text may be loaded dynamically
- pages contain advertising, tracking, and recommendation modules
- terms and `robots.txt` rules vary by outlet
- copying/exporting full article text can raise copyright concerns

The same inspection logic applies to news pages, but Wikipedia is a safer bridge before moving to more fragile sites.


## Fetch the Wikipedia Page

This is the static-scraping equivalent of Day 1's API request: send an HTTP request and receive text back.

The difference is the response format. The API returned JSON; the website returns HTML.


In [ ]:
wikipedia_url = "https://en.wikipedia.org/wiki/Digital_Services_Act"

wikipedia_headers = {
    "User-Agent": "methodsNET-VLOP-course/1.0 static scraping Wikipedia example"
}

# requests.get() fetches the raw browser-page HTML.
wiki_response = requests.get(wikipedia_url, headers=wikipedia_headers, timeout=30)
# raise_for_status() stops the workflow if the server returns an HTTP error.
wiki_response.raise_for_status()

wiki_html = wiki_response.text

print("Status:", wiki_response.status_code)
print("Characters of HTML:", len(wiki_html))
print(wiki_html[:500])


## Save the Raw Wikipedia HTML

Raw HTML is evidence. It lets us rerun parsing code without repeatedly contacting the website, and it preserves what the scraper saw at collection time.


In [ ]:
wiki_raw_path = raw_dir / "notebook_wikipedia_dsa_raw.html"
wiki_raw_path.write_text(wiki_html, encoding=wiki_response.encoding or "utf-8")

print("Saved raw Wikipedia HTML:", wiki_raw_path)


## Parse and Inspect the Wikipedia Page

A Wikipedia article page contains far more than article text. Before extracting, inspect broad structure: the title, main heading, paragraphs, headings, and links.


In [ ]:
# Parse the raw Wikipedia HTML into a searchable tree.
wiki_soup = BeautifulSoup(wiki_html, "html.parser")

# The HTML <title> is the browser-tab title. It is not always identical to the article heading.
print("HTML title:", wiki_soup.title.get_text(" ", strip=True))

# h1 is usually the visible article title.
main_heading = wiki_soup.select_one("h1")
print("Main heading:", main_heading.get_text(" ", strip=True) if main_heading else None)

# These broad counts include article content plus some non-article page elements.
print("Paragraph elements:", len(wiki_soup.select("p")))
print("Links:", len(wiki_soup.select("a[href]")))
print("h2/h3 headings:", len(wiki_soup.select("h2, h3")))


## Identify the Main Content Area

If we select all paragraphs from the whole page, we may capture navigation, footers, sidebars, or other non-article material.

A better first choice is to restrict extraction to the article content area. On Wikipedia, the parsed article body is usually inside `.mw-parser-output`.

This is a selector choice and should be documented. If Wikipedia changes its HTML structure, this selector might need revision.


In [ ]:
# Select the main parsed article content area.
content = wiki_soup.select_one(".mw-parser-output")

if content is None:
    print("Could not find article content area. Inspect the page structure again.")
else:
    print("Found article content area")


## Extract Article Paragraphs

Here the unit is not a repeated card. The page is a document, so we extract paragraphs from the main content area.

We also filter very short paragraphs. That filtering rule is a methodological choice: it changes what becomes part of the processed dataset.


In [ ]:
paragraph_rows = []

if content is not None:
    # "p" selects paragraph elements inside the article content area.
    for paragraph_number, paragraph in enumerate(content.select("p"), start=1):
        # get_text(" ", strip=True) extracts readable paragraph text.
        paragraph_text = paragraph.get_text(" ", strip=True)

        # Skip empty or very short paragraphs in this teaching example.
        # In a real project, this filtering rule should be documented.
        if len(paragraph_text) < 40:
            continue

        paragraph_rows.append(
            {
                "source_url": wikipedia_url,
                "paragraph_number": paragraph_number,
                "text": paragraph_text,
                "n_characters": len(paragraph_text),
                "n_words": len(paragraph_text.split()),
            }
        )

wiki_paragraphs_df = pd.DataFrame(paragraph_rows)
wiki_paragraphs_df.head()


## Extract Headings as Document Structure

Headings give a rough map of the article. This is different from extracting paragraph text; it tells us how the document is organized.


In [ ]:
heading_rows = []

if content is not None:
    for heading in content.select("h2, h3"):
        heading_rows.append(
            {
                "source_url": wikipedia_url,
                "level": heading.name,
                "heading": heading.get_text(" ", strip=True),
            }
        )

wiki_headings_df = pd.DataFrame(heading_rows)
wiki_headings_df.head(20)


## Extract and Classify Links

Links are analytically different from paragraphs. They can point to other Wikipedia pages, external sources, same-page anchors, files, or other resources.

We classify links because those types should not be interpreted as the same thing.


In [ ]:
wiki_link_rows = []

if content is not None:
    for link in content.select("a[href]"):
        href_raw = link.get("href")
        link_text = link.get_text(" ", strip=True)

        # Skip empty links or links without readable anchor text.
        if not href_raw or not link_text:
            continue

        # urljoin turns relative Wikipedia links into absolute URLs.
        href_absolute = urljoin(wikipedia_url, href_raw)

        if href_raw.startswith("/wiki/"):
            link_type = "internal_wikipedia_link"
        elif href_raw.startswith("http") or href_raw.startswith("//"):
            link_type = "external_link"
        elif href_raw.startswith("#"):
            link_type = "same_page_anchor"
        else:
            link_type = "other"

        wiki_link_rows.append(
            {
                "source_url": wikipedia_url,
                "link_text": link_text,
                "href_raw": href_raw,
                "href_absolute": href_absolute,
                "link_type": link_type,
            }
        )

wiki_links_df = pd.DataFrame(wiki_link_rows)
wiki_links_df.head(10)


In [ ]:
# Quick audit: how many links of each type did we collect?
wiki_links_df["link_type"].value_counts()


## Save the Wikipedia Processed Outputs

The Wikipedia example produces several processed tables because the page has several possible units:

- one row per paragraph
- one row per heading
- one row per link

This is different from the Quotes example, where the main output was one row per quote.


In [ ]:
wiki_paragraphs_path = processed_dir / "notebook_wikipedia_dsa_paragraphs.csv"
wiki_headings_path = processed_dir / "notebook_wikipedia_dsa_headings.csv"
wiki_links_path = processed_dir / "notebook_wikipedia_dsa_links.csv"

wiki_paragraphs_df.to_csv(wiki_paragraphs_path, index=False)
wiki_headings_df.to_csv(wiki_headings_path, index=False)
wiki_links_df.to_csv(wiki_links_path, index=False)

print(wiki_paragraphs_path)
print(wiki_headings_path)
print(wiki_links_path)


## What the Wikipedia Example Adds

The Wikipedia API example from Day 1 taught structured API responses: endpoint, parameters, JSON, IDs, pagination, and page-level requests.

The Wikipedia HTML example teaches page-structure extraction: a page may contain multiple possible units of analysis.

For a document-like page, ask:

- Do I want paragraphs, headings, links, tables, references, or images?
- What selector defines the main content area?
- What page elements should be excluded?
- Which filtering rules did I apply?
- Would an API be more stable and reproducible than scraping the page HTML?


# Clean Practice Page: Quotes to Scrape

Wikipedia is useful for continuity, but it is a document page. Many scraping tasks instead involve repeated records: cards, list items, products, posts, search results, or course rows.

`quotes.toscrape.com` is a small practice site designed for this. We keep it because it makes the repeated-record pattern easy to see before the MethodsNET exercise.


In [ ]:
URL = "https://quotes.toscrape.com/"
USER_AGENT = "methodsNET-VLOP-course/1.0_static_scraping_walkthrough"


The `User-Agent` identifies the script. It should be meaningful and honest.

# Check robots.txt

`robots.txt` is not full legal or ethical permission, but it is a first access
signal.

In [ ]:
parsed = urlparse(URL)
robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"

rp = RobotFileParser()
rp.set_url(robots_url)
rp.read()

allowed = rp.can_fetch(USER_AGENT, URL)

print("robots.txt URL:", robots_url)
print("Can fetch classroom URL?", allowed)


::: {.callout-question}
Why is `robots.txt` not the same as full permission to collect data?
:::

# Fetch the Page

`requests.get()` retrieves the raw HTML returned by the server. It does not
click, scroll, or execute JavaScript.

In [ ]:
# requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
response = requests.get(URL, headers={"User-Agent": USER_AGENT}, timeout=30)
# raise_for_status() turns HTTP error responses (such as 403, 404, or 429) into visible Python errors.
response.raise_for_status()

html = response.text

print("HTTP status:", response.status_code)
print("Characters of HTML:", len(html))
print(html[:500])


# Save Raw HTML

Save raw HTML before parsing. This lets us rerun extraction code without
contacting the site again and preserves evidence of what the scraper saw.

In [ ]:
raw_html_path = raw_dir / "notebook_quotes_raw.html"
raw_html_path.write_text(html, encoding=response.encoding or "utf-8")

print("Saved raw HTML:", raw_html_path)


## What Happens Between `requests` and BeautifulSoup?

The `requests` package gives us `response.text`: a long string containing the HTML returned by the server.

At that stage, Python has not yet extracted quotes, authors, links, or tags. It only has page source text.

`BeautifulSoup(html, "html.parser")` changes the task: it parses that string into a searchable tree. After that, we can ask questions such as:

- Which elements have class `.quote`?
- Which links have an `href` attribute?
- What visible text is inside `.author`?
- Which URL is stored inside a link's `href` attribute?

Fetching and parsing are separate steps. Saving the raw HTML before parsing preserves evidence of what the scraper actually saw.


# Parse and Inspect HTML

In [ ]:
# BeautifulSoup turns HTML text into a searchable parse tree.
soup = BeautifulSoup(html, "html.parser")

# prettify() prints nested HTML with indentation, which makes the page structure easier to inspect.
print(soup.prettify()[:1500])


Quick sanity checks:

In [ ]:
# get_text(" ", strip=True) extracts visible text, joins internal whitespace with spaces, and removes leading/trailing whitespace.
print("Page title:", soup.title.get_text(strip=True))
# select() returns a list of all elements matching a CSS selector.
print("Number of links:", len(soup.select("a[href]")))
print("Number of quote blocks:", len(soup.select(".quote")))


Open the same page in a browser and inspect one quote. Connect the HTML
structure to the selectors `.quote`, `.text`, `.author`, and `.tags`.

# Inspect One Record Before Looping

A good scraper starts with one record. If one record is wrong, the loop will
only repeat the mistake.

In [ ]:
# select_one() returns the first element matching a CSS selector, or None if nothing matches.
first_quote = soup.select_one(".quote")

# prettify() prints nested HTML with indentation, which makes the page structure easier to inspect.
print(first_quote.prettify())


Select fields inside that one record:

In [ ]:
# select_one() returns the first element matching a CSS selector, or None if nothing matches.
print("Quote text element:", first_quote.select_one(".text"))
print("Author element:", first_quote.select_one(".author"))
# select() returns a list of all elements matching a CSS selector.
print("Tag link elements:", first_quote.select(".tags a.tag"))
print("Author profile link:", first_quote.select_one("span a[href]"))


## Why Inspect One Record Before Looping?

Scraping usually fails in small ways before it fails in large ways: the selector is too broad, too narrow, or points to the wrong level of the HTML.

That is why the walkthrough first inspects one `.quote` block. Once one record is understood, the loop repeats the same extraction for every quote block.

The key methodological decision is the record container. Here, `.quote` means: one row in the output table represents one quote block on the page.


# Extract Repeated Records

Each `.quote` block is one repeated record container.

In [ ]:
rows = []

# select() returns a list of all elements matching a CSS selector.
for quote_block in soup.select(".quote"):
    # select_one() returns the first element matching a CSS selector, or None if nothing matches.
    text = quote_block.select_one(".text")
    author = quote_block.select_one(".author")
    tag_links = quote_block.select(".tags a.tag")
    author_link = quote_block.select_one("span a[href]")

    row = {
        # get_text(" ", strip=True) extracts visible text, joins internal whitespace with spaces, and removes leading/trailing whitespace.
        "quote": text.get_text(" ", strip=True) if text else None,
        "author": author.get_text(" ", strip=True) if author else None,
        # .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
        "author_href_raw": author_link.get("href") if author_link else None,
        "author_href_absolute": (
            # urljoin() turns a relative URL such as /page/2/ into a full absolute URL using the page URL as context.
            urljoin(URL, author_link.get("href")) if author_link else None
        ),
        "tags": "|".join(tag.get_text(" ", strip=True) for tag in tag_links),
        "tag_count": len(tag_links),
    }

    rows.append(row)

# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
quotes_df = pd.DataFrame(rows)
quotes_df.head()


## Why Extract Links Separately?

Links are often analytically different from visible text.

A link has:

- visible anchor text, such as `about`
- an `href` attribute, such as `/author/Albert-Einstein`
- a role on the page, such as author profile, tag page, or pagination

The script classifies links because not every link is part of the same data object. Navigation links, author links, tag links, and next-page links mean different things.


::: {.callout-important}
The selector decides the unit of analysis. Here, `.quote` means one row is one
quote block.
:::

# Extract and Classify Links

All links are `<a>` elements with an `href` attribute.

In [ ]:
links = []

# select() returns a list of all elements matching a CSS selector.
for link in soup.select("a[href]"):
    # .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
    href_raw = link.get("href")
    # urljoin() turns a relative URL such as /page/2/ into a full absolute URL using the page URL as context.
    href_absolute = urljoin(URL, href_raw)

    if href_raw.startswith("/author/"):
        link_type = "author_profile"
    elif href_raw.startswith("/tag/"):
        link_type = "tag_page"
    elif href_raw.startswith("/page/"):
        link_type = "pagination"
    else:
        link_type = "other"

    links.append(
        {
            # get_text(" ", strip=True) extracts visible text, joins internal whitespace with spaces, and removes leading/trailing whitespace.
            "link_text": link.get_text(" ", strip=True),
            "href_raw": href_raw,
            "href_absolute": href_absolute,
            "link_type": link_type,
        }
    )

# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
links_df = pd.DataFrame(links)
links_df.head(10)


## Why Make a Separate Tag Table?

A quote can have multiple tags. That creates a design choice.

One table can store tags as a single combined string, for example `life|truth|books`. That is simple and easy to save as CSV.

A separate long table stores one row per quote-tag pair. That is better for counting tags, filtering by tags, or joining tags to other data.

This is not just a Python choice; it changes the shape of the dataset.


# Tags as a Separate Table

Sometimes repeated fields should become their own table.

In [ ]:
tag_rows = []

# select() returns a list of all elements matching a CSS selector.
for quote_number, quote_block in enumerate(soup.select(".quote"), start=1):
    # select_one() returns the first element matching a CSS selector, or None if nothing matches.
    text = quote_block.select_one(".text")
    # get_text(" ", strip=True) extracts visible text, joins internal whitespace with spaces, and removes leading/trailing whitespace.
    quote_text = text.get_text(" ", strip=True) if text else None

    for tag_link in quote_block.select(".tags a.tag"):
        tag_rows.append(
            {
                "quote_number": quote_number,
                "quote": quote_text,
                "tag": tag_link.get_text(" ", strip=True),
                # .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
                "tag_url": urljoin(URL, tag_link.get("href")),
            }
        )

# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
tags_df = pd.DataFrame(tag_rows)
tags_df.head()


# Save Processed Tables

In [ ]:
quotes_path = processed_dir / "notebook_quotes.csv"
links_path = processed_dir / "notebook_quote_links.csv"
tags_path = processed_dir / "notebook_quote_tags.csv"

# to_csv(..., index=False) saves a DataFrame as a CSV file without adding pandas row numbers as a fake column.
quotes_df.to_csv(quotes_path, index=False)
links_df.to_csv(links_path, index=False)
tags_df.to_csv(tags_path, index=False)

print(quotes_path)
print(links_path)
print(tags_path)


# Exercise

For a short individual/pair exercise, use the MethodsNET course-list/detail example in `day_2_beautifulsoup_extraction_patterns.ipynb`.

The goal is to transfer the same static-scraping logic to a more realistic course website:

1. inspect the HTML before extracting;
2. identify the repeated record container;
3. extract links and visible text;
4. document which selectors assume a stable table or page structure;
5. compare the scaffolded exercise with the solution notebook after trying it.

Optional warm-up using the quotes page:

1. Extract the author profile URLs.
2. Extract the next-page URL.
3. Create a separate table of tags.
4. Add a column counting how many tags each quote has.
5. Save a short provenance note explaining the URL, selectors, and date.

# Key Takeaway

Scraping is not just "getting what is on a page".

It involves choices about access, selectors, units of analysis, missing data, raw evidence, and documentation.
